# Báo cáo baseline cuối: ResNet-50 và ConvNeXt-V2-Base

Ngày tạo: 20/10/2026. Mục tiêu: dựng lại bảng/ma trận từ artifacts baseline hoàn tất ngày 15/10, kiểm tra chúng khớp receipts tính lại NPZ. Đây là notebook tổng hợp báo cáo, không phải lần train hoặc lần chạy lại COCOeval. Lượt tính lại AP/calibration/counting được ghi trong `recomputed_metrics.json`; script đi kèm audit folder.

Input: campaign `vast4090-seed0-640-v1/baseline`, audit `final_metrics_review_20261020`. Output: bảng Markdown trong notebook và receipt tổng hợp dưới folder audit với run ID riêng. CPU, Conda pdp Python 3.10; không cần GPU/network. Không chọn policy trên test.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json, hashlib, uuid, sys, platform, subprocess
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
BASE = ROOT/'data/training/vast4090-seed0-640-v1/baseline'
AUDIT = ROOT/'data/training/validation/final_metrics_review_20261020'
run_id = str(uuid.uuid4())
now = lambda: datetime.now(ZoneInfo('Asia/Bangkok')).isoformat()
read = lambda p: json.loads(p.read_text())
record = dict(run_id=run_id, started_at=now(), status='RUNNING', python=sys.version,
              host=platform.node(), seed=0, purpose='report aggregation, not inference or metric recomputation',
              git_commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),
              git_dirty=bool(subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()))
summary=read(BASE/'summary.json')
assert summary['status']=='COMPLETED' and not summary['dry_run']
verified=read(AUDIT/'recomputed_metrics.json')
snapshot=read(ROOT/'docs/project/assets/final_backbone_metrics_2026-10-20.json')
for name,digest in snapshot['sources'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==digest, name
record['source_sha256']=snapshot['sources']
print(json.dumps({k:v for k,v in record.items() if k!='source_sha256'},ensure_ascii=False,indent=2))
print('Campaign completed:',summary['finished_at'])


{
  "run_id": "8f4b163c-4f5a-411b-8993-e553f5c95e95",
  "started_at": "2026-10-20T10:32:09.025286+07:00",
  "status": "RUNNING",
  "python": "3.10.22 | packaged by conda-forge | (main, Oct  6 2026, 23:02:24) [GCC 14.4.0]",
  "host": "archlinux",
  "seed": 0,
  "purpose": "report aggregation, not inference or metric recomputation",
  "git_commit": "1e81302f9f8da8aacefe208df24a5d42e6ab10a8",
  "git_dirty": true
}
Campaign completed: 2026-10-15T19:28:27+07:00


In [2]:
table=['| Model | Task | AP50 (%) | AP[.50:.95] (%) | cAcc (%) | ACD |',
       '|---|---:|---:|---:|---:|---:|']
for exp in ['EXP-B1','EXP-B2']:
    d=read(BASE/exp/'metrics_cl_test.json'); c=read(BASE/exp/'metrics_count_test.json')
    assert not c['oracle_enabled']
    for t in range(1,6):
        key=str(t); evidence=verified['experiments'][exp]['stages'][key]
        assert evidence['counting_matches']
        assert evidence['counting']==c['stages'][key]['test']
        ap=d['stages'][key]['m1']['mAP_A']; count=c['stages'][key]['test']['overall']
        table.append(f"| {exp} | {t} | {100*ap['AP50']:.2f} | {100*ap['AP']:.2f} | {100*count['cAcc']:.2f} | {count['ACD']:.4f} |")
    final=verified['experiments'][exp]['stages']['5']
    assert final['detection_matches'] and final['val_calibration_matches']
    assert final['detection']==d['stages']['5']['m1']
display(Markdown('\n'.join(table)))


| Model | Task | AP50 (%) | AP[.50:.95] (%) | cAcc (%) | ACD |
|---|---:|---:|---:|---:|---:|
| EXP-B1 | 1 | 83.16 | 70.07 | 17.52 | 2.3025 |
| EXP-B1 | 2 | 79.14 | 66.50 | 7.46 | 3.7938 |
| EXP-B1 | 3 | 73.94 | 61.82 | 4.60 | 5.1411 |
| EXP-B1 | 4 | 67.99 | 56.61 | 3.18 | 6.8176 |
| EXP-B1 | 5 | 62.33 | 51.55 | 2.58 | 8.8311 |
| EXP-B2 | 1 | 98.35 | 83.70 | 73.16 | 0.3678 |
| EXP-B2 | 2 | 97.54 | 82.98 | 53.32 | 0.8231 |
| EXP-B2 | 3 | 94.66 | 80.45 | 29.97 | 1.9207 |
| EXP-B2 | 4 | 89.72 | 76.10 | 22.29 | 2.9980 |
| EXP-B2 | 5 | 84.40 | 71.37 | 21.74 | 3.9187 |

In [3]:
from autocheckout.cl_metrics import matrix_forgetting
for exp in ['EXP-B1','EXP-B2']:
    d=read(BASE/exp/'metrics_cl_test.json')
    matrix={int(t):{int(g):v for g,v in row.items()} for t,row in d['matrix'].items()}
    forgetting=matrix_forgetting(matrix)
    assert abs(forgetting['average']-d['forgetting']['average'])<1e-12
    lines=[f'### {exp}: AP50 (%)', '| Stage | Nhóm 1 | Nhóm 2 | Nhóm 3 | Nhóm 4 | Nhóm 5 |', '|---|---:|---:|---:|---:|---:|']
    for t,row in matrix.items():
        values=[f'{100*row[g]:.2f}' if g in row else '—' for g in range(1,6)]
        lines.append('| '+str(t)+' | '+' | '.join(values)+' |')
    lines.append(f"\nForgetting trung bình: {100*forgetting['average']:.2f} điểm AP50.")
    display(Markdown('\n'.join(lines)))


### EXP-B1: AP50 (%)
| Stage | Nhóm 1 | Nhóm 2 | Nhóm 3 | Nhóm 4 | Nhóm 5 |
|---|---:|---:|---:|---:|---:|
| 1 | 83.16 | — | — | — | — |
| 2 | 83.58 | 65.72 | — | — | — |
| 3 | 81.73 | 63.99 | 61.08 | — | — |
| 4 | 78.48 | 59.88 | 58.31 | 54.43 | — |
| 5 | 75.39 | 56.01 | 54.32 | 52.53 | 47.59 |

Forgetting trung bình: 6.63 điểm AP50.

### EXP-B2: AP50 (%)
| Stage | Nhóm 1 | Nhóm 2 | Nhóm 3 | Nhóm 4 | Nhóm 5 |
|---|---:|---:|---:|---:|---:|
| 1 | 98.35 | — | — | — | — |
| 2 | 98.22 | 95.24 | — | — | — |
| 3 | 97.68 | 93.02 | 85.50 | — | — |
| 4 | 96.51 | 86.45 | 81.60 | 76.40 | — |
| 5 | 94.30 | 77.91 | 74.55 | 74.07 | 74.70 |

Forgetting trung bình: 8.67 điểm AP50.

## Diễn giải và giới hạn

ConvNeXt có AP/counting cuối tốt hơn, nhưng forgetting trung bình lớn hơn. AP50 giảm qua stage không hoàn toàn là forgetting vì tập lớp đánh giá tăng. cAcc yêu cầu đúng toàn bộ số lượng từng SKU trong ảnh. Paper PDP dùng AP50 trong bảng COCO multi-step; không so 59,4 trực tiếp với AP[.50:.95] của RPC và không tuyên bố vượt paper trên dataset khác.

Hai final checkpoints đã audit finite/200 prototype khác zero. `EXP-B2/task_5/progress.json` vẫn ghi RUNNING/25.213 dù final artifacts hoàn tất: giữ nguyên lịch sử, chưa chốt nguyên nhân. Một seed, chưa kiểm chứng hội tụ; không retune bằng test. Xem báo cáo `docs/project/FINAL_BACKBONE_COMPARISON_2026-10-20.md`.


In [4]:
record.update(status='COMPLETED', finished_at=now(), experiments=['EXP-B1','EXP-B2'], stages_per_model=5,
              validation='source hashes, counting receipts, final AP receipts and forgetting formula',
              limitations=['does not rerun training/inference/COCOeval', 'single seed', 'no convergence claim'])
destination=AUDIT/'report_notebooks'/run_id
destination.mkdir(parents=True,exist_ok=True)
(destination/'summary.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in record.items() if k!='source_sha256'},ensure_ascii=False,indent=2))
print('Receipt:',destination.relative_to(ROOT))


{
  "run_id": "8f4b163c-4f5a-411b-8993-e553f5c95e95",
  "started_at": "2026-10-20T10:32:09.025286+07:00",
  "status": "COMPLETED",
  "python": "3.10.22 | packaged by conda-forge | (main, Oct  6 2026, 23:02:24) [GCC 14.4.0]",
  "host": "archlinux",
  "seed": 0,
  "purpose": "report aggregation, not inference or metric recomputation",
  "git_commit": "1e81302f9f8da8aacefe208df24a5d42e6ab10a8",
  "git_dirty": true,
  "finished_at": "2026-10-20T10:32:09.078687+07:00",
  "experiments": [
    "EXP-B1",
    "EXP-B2"
  ],
  "stages_per_model": 5,
  "validation": "source hashes, counting receipts, final AP receipts and forgetting formula",
  "limitations": [
    "does not rerun training/inference/COCOeval",
    "single seed",
    "no convergence claim"
  ]
}
Receipt: data/training/validation/final_metrics_review_20261020/report_notebooks/8f4b163c-4f5a-411b-8993-e553f5c95e95
